# Feature-set comparison

Cross-validate every named feature set against a target with a fixed set of model templates, to see which NMR representation carries the most signal.

**Input:** a featurized dataset (output of `create_features_nmr.ipynb`).  
**Output:** a results table + comparison bar charts.

The feature sets come from [`nmrlib.features.feature_sets`](nmrlib/features.py) and the CV loop from [`nmrlib.models.compare_feature_sets`](nmrlib/models.py).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from nmrlib import load_dataset, compare_feature_sets
from nmrlib.features import feature_sets
from nmrlib.models import default_models

## Available datasets

Registry from [`nmrlib.data`](nmrlib/data.py). Descriptions are hardcoded labels; `found_in` is checked live against the filesystem. The column listing printed after loading (below) is the sanity check that a dataset's contents actually match its description.

In [ ]:
from nmrlib import describe_datasets

describe_datasets()

## Config

In [ ]:
dataset = "ids_1k_featurized"   # registry name or path to a featurized .pkl
target_col = "gap_ev"           # gap_ev | log_p_rdkit | homo_ev | lumo_ev
seed = 1
n_splits = 10
test_frac = 0.15

## Load

In [ ]:
df = load_dataset(dataset)
sets = feature_sets(df)
print(f"{len(sets)} feature sets available for target {target_col!r}:")
for name, cols in sets.items():
    print(f"  {name:<28} {len(cols)} cols")
print(f"Columns ({len(df.columns)}): " + ", ".join(map(str, df.columns)))

## Compare

In [ ]:
results_df = compare_feature_sets(
    df, target_col, sets,
    models=default_models(seed),
    n_splits=n_splits, test_frac=test_frac, seed=seed,
)
results_df.sort_values("Test R²", ascending=False).reset_index(drop=True)

## Plot

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=results_df, x="Feature Set", y="Test RMSE", hue="Model")
plt.xticks(rotation=45, ha='right')
plt.title("Test RMSE by Feature Set and Model (lower is better)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=results_df, x="Feature Set", y="Test R²", hue="Model")
plt.xticks(rotation=45, ha='right')
plt.title("Test R² by Feature Set and Model (higher is better)")
plt.tight_layout()
plt.show()